# Uber Fare Prediction — Model Training & Comparison (K-Fold Cross-Validation)

This is a copy of `02_model_trainingnew.ipynb` extended with **K-Fold Cross-Validation** for both training and testing every model: **Linear Regression**, **Multiple Regression** (OLS), **Ridge**, **Lasso**, and **XGBoost**. Instead of relying on a single 80/20 split, each model is trained and evaluated across 5 folds, and the fold-wise MAE/RMSE/R² are averaged for a more robust performance estimate. A final model is then re-fit on the full 80% training split (as in the original notebook) for the sample fare prediction at the end.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

pd.set_option('display.max_columns', None)

## 2. Load Data

In [ ]:
data = pd.read_csv('processed_data.csv')
print("Shape:", data.shape)
data.head()

## 3. Basic Checks

In [ ]:
print("Missing values:", data.isnull().sum().sum())
print("\nData types:\n", data.dtypes.value_counts())

## 4. Feature / Target Split & Train-Test Split

An 80/20 hold-out split is still kept for the final sample-fare prediction at the end of the notebook, but model evaluation itself now happens via K-Fold CV on the training portion (see Section 6 onward).

In [ ]:
X = data.drop(columns=['Fare (Target)'])
y = data['Fare (Target)']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 5. K-Fold Setup & Evaluation Helpers

`KFold(n_splits=5, shuffle=True, random_state=42)` splits `X_train`/`y_train` into 5 folds. For each model, we loop over the folds: fit on 4 folds, evaluate (MAE/RMSE/R²) on the held-out fold, then average the metrics across all 5 folds. This gives a more reliable performance estimate than a single train/test split, since it isn't dependent on which rows happened to land in the test set.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = {}       # averaged K-Fold metrics per model (for comparison table/plots)
holdout_results = {}  # metrics on the untouched X_test/y_test, using a model refit on all of X_train

def run_kfold(name, model_builder, X_data, y_data, is_statsmodels=False):
    """
    model_builder: a zero-arg function returning a *fresh* unfitted model instance
                   (fresh instance per fold avoids leaking state between folds).
    is_statsmodels: True for the OLS model, which needs sm.add_constant() and a
                     different fit/predict call signature than sklearn estimators.
    """
    fold_mae, fold_rmse, fold_r2 = [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_data), start=1):
        X_tr, X_val = X_data.iloc[train_idx], X_data.iloc[val_idx]
        y_tr, y_val = y_data.iloc[train_idx], y_data.iloc[val_idx]

        if is_statsmodels:
            X_tr_sm = sm.add_constant(X_tr.astype(float))
            X_val_sm = sm.add_constant(X_val.astype(float), has_constant='add')
            fold_model = sm.OLS(y_tr, X_tr_sm).fit()
            y_pred = fold_model.predict(X_val_sm)
        else:
            fold_model = model_builder()
            fold_model.fit(X_tr, y_tr)
            y_pred = fold_model.predict(X_val)

        mae = mean_absolute_error(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        r2 = r2_score(y_val, y_pred)

        fold_mae.append(mae)
        fold_rmse.append(rmse)
        fold_r2.append(r2)
        print(f"  Fold {fold_idx}: MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}")

    avg_mae, avg_rmse, avg_r2 = np.mean(fold_mae), np.mean(fold_rmse), np.mean(fold_r2)
    results[name] = {'MAE': avg_mae, 'RMSE': avg_rmse, 'R2': avg_r2}
    print(f"{name} -- 5-Fold Average")
    print(f"  MAE : {avg_mae:.2f}")
    print(f"  RMSE: {avg_rmse:.2f}")
    print(f"  R2  : {avg_r2:.4f}\n")

def evaluate_holdout(name, model, y_true, y_pred):
    """Metrics on the untouched X_test/y_test hold-out set (used for the final comparison table)."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    holdout_results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"{name} -- Held-out Test Set")
    print(f"  MAE : {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R2  : {r2:.4f}")

## 6. Linear Regression -- K-Fold CV, then Final Fit

In [ ]:
print("Linear Regression -- 5-Fold Cross-Validation")
run_kfold('Linear Regression', lambda: LinearRegression(), X_train, y_train)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
evaluate_holdout('Linear Regression', lr_model, y_test, lr_pred)

## 7. Multiple Regression (OLS) -- K-Fold CV, then Final Fit

The K-Fold loop refits a fresh `sm.OLS` per fold (statsmodels doesn't implement the sklearn estimator API, so it can't be passed to `model_builder` the same way). The full statistical summary (coefficients, p-values, confidence intervals) is still printed from the final model fit on the full 80% training split, exactly as in the original notebook.

In [ ]:
print("Multiple Regression (OLS) -- 5-Fold Cross-Validation")
run_kfold('Multiple Regression (OLS)', None, X_train, y_train, is_statsmodels=True)

X_train_sm = sm.add_constant(X_train.astype(float))
X_test_sm = sm.add_constant(X_test.astype(float), has_constant='add')

mr_model = sm.OLS(y_train, X_train_sm).fit()
print(mr_model.summary())

In [ ]:
mr_pred = mr_model.predict(X_test_sm)
evaluate_holdout('Multiple Regression (OLS)', mr_model, y_test, mr_pred)

## 8. Ridge Regression -- K-Fold CV, then Final Fit

In [ ]:
print("Ridge Regression -- 5-Fold Cross-Validation")
run_kfold('Ridge Regression', lambda: Ridge(alpha=1.0, random_state=42), X_train, y_train)

ridge_model = Ridge(alpha=1.0, random_state=42)
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
evaluate_holdout('Ridge Regression', ridge_model, y_test, ridge_pred)

## 9. Lasso Regression -- K-Fold CV, then Final Fit

In [ ]:
print("Lasso Regression -- 5-Fold Cross-Validation")
run_kfold('Lasso Regression', lambda: Lasso(alpha=1.0, random_state=42, max_iter=5000), X_train, y_train)

lasso_model = Lasso(alpha=1.0, random_state=42, max_iter=5000)
lasso_model.fit(X_train, y_train)
lasso_pred = lasso_model.predict(X_test)
evaluate_holdout('Lasso Regression', lasso_model, y_test, lasso_pred)

### Features Lasso eliminated (coefficient shrunk to zero)

In [ ]:
lasso_coefs = pd.Series(lasso_model.coef_, index=X_train.columns)
eliminated = lasso_coefs[lasso_coefs == 0]
print(f"Lasso eliminated {len(eliminated)} of {len(lasso_coefs)} features:")
print(eliminated.index.tolist())

## 10. XGBoost Regression -- K-Fold CV, then Final Fit

In [ ]:
def build_xgb():
    return XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

print("XGBoost -- 5-Fold Cross-Validation")
run_kfold('XGBoost', build_xgb, X_train, y_train)

xgb_model = build_xgb()
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
evaluate_holdout('XGBoost', xgb_model, y_test, xgb_pred)

## 11. Model Comparison

Two side-by-side views: the **5-Fold CV average** (robust estimate, from Sections 6-10) and the **held-out test set** result (the classic single-split comparison, as in the original notebook).

In [ ]:
cv_comparison_df = pd.DataFrame(results).T.sort_values(by='R2', ascending=False)
print("5-Fold CV Average (on training data)")
cv_comparison_df

In [ ]:
holdout_comparison_df = pd.DataFrame(holdout_results).T.sort_values(by='R2', ascending=False)
print("Held-out Test Set")
holdout_comparison_df

### Visual comparison of R², MAE, and RMSE across models (5-Fold CV average)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cv_comparison_df['R2'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('R² Score by Model (5-Fold Avg)')
axes[0].set_ylabel('R²')
axes[0].tick_params(axis='x', rotation=45)

cv_comparison_df['MAE'].plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('MAE by Model (5-Fold Avg)')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=45)

cv_comparison_df['RMSE'].plot(kind='bar', ax=axes[2], color='indianred')
axes[2].set_title('RMSE by Model (5-Fold Avg)')
axes[2].set_ylabel('RMSE')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Summary

- The **5-Fold CV average** metrics are generally a more trustworthy estimate of how each model will perform on unseen data than the single held-out split, since they're averaged over 5 different train/validation partitions.
- If the CV average and the held-out result diverge a lot for a given model, that's a sign the single 80/20 split was an unusually easy/hard split for that model.
- **Linear Regression** and **Multiple Regression (OLS)** still track each other closely, as expected -- same underlying method.
- **XGBoost** is expected to remain the strongest performer if the Fare relationship is non-linear, consistent with the original notebook's findings.

## 12. Sample Fare Prediction — Bike vs Sedan (15 km), All Models

Same synthetic 15 km trip as the original notebook, predicted with each model's **final fit** (trained on the full 80% training split, i.e. the models from Sections 6-10, not any single fold's model).

In [ ]:
# Create a row with all features initialised as 0, matching X_train's columns exactly
bike_prediction = pd.DataFrame(0, index=[0], columns=X_train.columns)

# Numeric features
bike_prediction['Distance (km)'] = 15
bike_prediction['Estimated Duration (min)'] = 30

# Normal conditions
bike_prediction['Weather Condition'] = 0        # Clear
bike_prediction['Rainfall Intensity'] = 0       # No Rain
bike_prediction['Traffic Level'] = 0            # Low
bike_prediction['Temperature Level'] = 1        # Pleasant
bike_prediction['Holiday'] = 0                  # No Holiday
bike_prediction['Day Type'] = 0                 # Weekday
bike_prediction['Number of Stops Added'] = 0

# Normal categorical values
bike_prediction['Pickup Area Type_Residential'] = 1
bike_prediction['Drop Area Type_Residential'] = 1
bike_prediction['Time Slot_Midday'] = 1
bike_prediction['Busy Day_Normal'] = 1

# Vehicle type = Bike
bike_prediction['Vehicle Type_Bike'] = 1

# Sedan version of the same trip
sedan_prediction = bike_prediction.copy()
sedan_prediction['Vehicle Type_Bike'] = 0
sedan_prediction['Vehicle Type_Sedan'] = 1

# statsmodels (Multiple Regression) needs the constant column added, same as during training
bike_prediction_sm = sm.add_constant(bike_prediction.astype(float), has_constant='add')
sedan_prediction_sm = sm.add_constant(sedan_prediction.astype(float), has_constant='add')

# Predict with every trained model
sample_predictions = {
    'Linear Regression': (
        lr_model.predict(bike_prediction)[0],
        lr_model.predict(sedan_prediction)[0]
    ),
    'Multiple Regression (OLS)': (
        mr_model.predict(bike_prediction_sm)[0],
        mr_model.predict(sedan_prediction_sm)[0]
    ),
    'Ridge Regression': (
        ridge_model.predict(bike_prediction)[0],
        ridge_model.predict(sedan_prediction)[0]
    ),
    'Lasso Regression': (
        lasso_model.predict(bike_prediction)[0],
        lasso_model.predict(sedan_prediction)[0]
    ),
    'XGBoost': (
        xgb_model.predict(bike_prediction)[0],
        xgb_model.predict(sedan_prediction)[0]
    ),
}

sample_df = pd.DataFrame(sample_predictions, index=['Predicted Bike Fare (₹)', 'Predicted Sedan Fare (₹)']).T
sample_df = sample_df.round(2)
sample_df